# 01 · El caso y los datos

**Bloque 1 de 4 · 45 minutos**

---

## 📧 La petición

> **De:** Dirección de Finanzas
> **Para:** Analista de Control Presupuestal
> **Asunto:** Comité del viernes — Hermosillo
>
> *"Necesito entender por qué el gasto de Hermosillo se disparó en el Q2.
> Quiero saber si es algo puntual o una tendencia, y qué centros de costo lo
> explican. Mándame algo que pueda proyectar."*

---

## Lo primero es traducir la petición

Así llegan las peticiones reales: sin especificar tablas, ni columnas, ni
periodos exactos. Antes de escribir una línea de código, hay que convertir
ese párrafo en preguntas que los datos puedan responder.

| Lo que dijo | Lo que hay que calcular | Con qué se responde |
|---|---|---|
| *"se disparó"* | **¿Cuánto?** La desviación contra presupuesto | Un número y su comparación |
| *"¿puntual o tendencia?"* | **¿Cuándo?** La evolución mes a mes | Una serie de tiempo |
| *"qué centros lo explican"* | **¿Dónde?** El desglose por centro de costo | Un ranking |

Esas tres preguntas son la estructura de tu reporte. Al final del bloque 3
vamos a volver a este correo para comprobar que las respondimos todas.

---

## El plan de los próximos 45 minutos

```
   3 archivos CSV crudos
            │
            ▼
   ① ¿abren?            ← codificación
   ② ¿son números?      ← formato de importes
   ③ ¿están completos?  ← duplicados
   ④ ¿se pueden unir?   ← llaves de texto
   ⑤ ¿qué es un error   ← criterio contable
      y qué no?
            │
            ▼
   datos_limpios.parquet  → listo para analizar
```

> 💡 **Cómo funciona este notebook**
> Las celdas marcadas 📖 las ejecuta y explica el instructor.
> Las marcadas ✏️ **TU TURNO** las resuelves tú, y abajo tienes una celda que
> te dice si acertaste. Si te atoras, no te preocupes: el bloque siguiente
> arranca de un archivo ya preparado.

---
## 📖 1.1 · Cargar el primer archivo

Empezamos por los hechos: el detalle del gasto real.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Que el notebook funcione lo mismo si lo abres desde la raíz o desde notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from utils.verificar import verificar

CRUDOS = RAIZ / "datos" / "crudos"

hechos = pd.read_csv(CRUDOS / "hechos_gasto.csv")

print(f"{len(hechos):,} filas")
hechos.head()

12,800 filas


,folio,fecha,planta,centro_costo,concepto,monto,cargado_en
0,CHI-202501-006264,2025-01-01,Chihuahua,Energía,Refacción prensa,"6.375,00",2026-07-01
1,CHI-202501-006291,2025-01-01,Chihuahua,nómina indirecta,Flete nacional,"9.066,75",2026-07-01
2,CHI-202501-006318,2025-01-01,Chihuahua,Fletes,Calibración de equipo,"7.912,50",2026-07-01
3,CHI-202501-006345,2025-01-01,Chihuahua,Calidad,Equipo de protección,"4.524,25",2026-07-01
4,CHI-202501-006372,2025-01-01,Chihuahua,Seguridad,Refacción prensa,"4.405,75",2026-07-01


---
## ✏️ 1.2 · TU TURNO · Cargar el presupuesto

Sin presupuesto no hay desviación que calcular. Cárgalo.

> 💡 **Pista:** es el mismo patrón de la celda anterior. Solo cambia el
> nombre del archivo: `presupuesto.csv`.

In [ ]:
presupuesto = ____________________________________

presupuesto.head()

In [ ]:
verificar("1.2", presupuesto)

---
## ✏️ 1.3 · TU TURNO · El archivo que no abre

Falta la tercera tabla: el catálogo de centros de costo, que nos dice quién
es responsable de cada uno.

Ejecuta esta celda. **Va a fallar**, y eso es parte del ejercicio.

In [ ]:
# Esta celda REVIENTA a propósito. Lee el error con calma.
dim_cc = pd.read_csv(CRUDOS / "dim_centro_costo.csv")

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf1 in position 58: invalid continuation byte

El error dice algo así:

```
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf3 in position 47
```

Traducido: *"intenté leer este archivo asumiendo UTF-8 y me topé con un byte
que en UTF-8 no significa nada"*.

### Por qué pasa esto todo el tiempo

Un archivo de texto es una secuencia de bytes. Para saber qué letra
representa cada byte hace falta saber **con qué codificación** se escribió.
Si te equivocas, o truena o —peor— te da letras corruptas en silencio.

| Si el archivo viene de… | usa `encoding=` |
|---|---|
| Excel en Windows en español | `"cp1252"` |
| SAP o un sistema antiguo | `"latin-1"` |
| Un export moderno, una API, la web | `"utf-8"` (es el default) |
| No sabes | prueba `"cp1252"` |
| Vas a **escribir** para Power BI o Excel | `"utf-8-sig"` |

Guarda esta tabla: la vamos a volver a usar en el bloque 3, cuando toque
exportar.

> 💡 **Pista:** este archivo lo exportó un Excel en Windows en español.
> Busca en la tabla qué codificación corresponde a ese origen.

In [ ]:
dim_cc = pd.read_csv(CRUDOS / "dim_centro_costo.csv", encoding="__________")

dim_cc

In [ ]:
verificar("1.3", dim_cc)

Comprueba que los acentos quedaron bien: debe decir **Energía** y
**Nómina Indirecta**. Si vieras `EnergÃ­a`, la codificación sería la
equivocada aunque el archivo hubiera abierto sin error.

> Nota: `"latin-1"` también funciona aquí. Son dos codificaciones casi
> idénticas y para estos acentos dan el mismo resultado. Las dos son
> correctas.

---
## 📖 1.4 · Los importes no son números

Ya tenemos las tres tablas. Antes de sumar nada, hay que mirar de qué tipo
es cada columna.

In [ ]:
hechos.info()

<class 'pandas.DataFrame'>
RangeIndex: 12800 entries, 0 to 12799
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   folio         12800 non-null  str  
 1   fecha         12800 non-null  str  
 2   planta        12800 non-null  str  
 3   centro_costo  12800 non-null  str  
 4   concepto      12800 non-null  str  
 5   monto         12800 non-null  str  
 6   cargado_en    12800 non-null  str  
dtypes: str(7)
memory usage: 1.7 MB


Fíjate en la columna `monto`: dice **`str`** — es decir, **texto**.

Eso significa que pandas no la está tratando como número. Y si intentas
sumarla, no obtienes lo que crees:

In [ ]:
print("Un valor de monto:", repr(hechos["monto"].iloc[0]))
print()
print("Lo que pasa si 'sumas' texto (concatena, no suma):")
print(repr(hechos["monto"].head(3).sum())[:80])

Un valor de monto: '6.375,00'

Lo que pasa si 'sumas' texto (concatena, no suma):
'6.375,009.066,757.912,50'


El culpable es el formato: `1.234.567,50`. Así escribe los números un Excel
en español — punto para los miles, coma para los decimales. Python espera lo
contrario.

**Esto es peligroso precisamente porque no truena.** Un total mal calculado
se ve como un total.

---
## ✏️ 1.5 · TU TURNO · Convertir los importes a número

Hay que quitar los separadores de miles y cambiar la coma decimal por punto,
en ese orden, y luego convertir a número.

> 💡 **Pista:** mira el valor tal como viene: `'1.234.567,50'`. Estorban dos
> símbolos y hacen trabajos distintos: uno separa miles, el otro separa
> decimales. Piensa qué carácter va en cada blanco.

In [ ]:
hechos["monto"] = (
    hechos["monto"]
    .str.replace("___", "", regex=False)      # fuera el separador de miles
    .str.replace("___", "___", regex=False)   # la coma decimal se vuelve punto
    .astype(float)
)

print(f"Gasto total: ${hechos['monto'].sum():,.2f}")
hechos[["folio", "monto"]].head(3)

In [ ]:
verificar("1.5", hechos)

El presupuesto tiene el mismo problema. Lo arreglamos igual — esta vez te lo
damos hecho para no repetir el ejercicio.

In [ ]:
presupuesto["monto_presupuesto"] = (
    presupuesto["monto_presupuesto"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print(f"Presupuesto total: ${presupuesto['monto_presupuesto'].sum():,.2f}")

Presupuesto total: $48,260,000.00


---
## 📖 1.6 · ¿Están completos los datos, o de más?

Un asiento contable debería aparecer una sola vez. Vamos a ver si es el caso.

In [ ]:
print(f"Filas en el archivo:  {len(hechos):,}")
print(f"Folios distintos:     {hechos['folio'].nunique():,}")
print(f"Diferencia:           {len(hechos) - hechos['folio'].nunique():,}")

Filas en el archivo:  12,800
Folios distintos:     12,000
Diferencia:           800


Hay **800 folios repetidos**. Veamos uno:

In [ ]:
repetido = hechos[hechos["folio"].duplicated(keep=False)]["folio"].iloc[0]
hechos[hechos["folio"] == repetido]

,folio,fecha,planta,centro_costo,concepto,monto,cargado_en
9771,HER-202604-000027,2026-04-01,Hermosillo,Mantenimiento,Flete nacional,4868.5,2026-07-01
9772,HER-202604-000027,2026-04-01,Hermosillo,Mantenimiento,Flete nacional,4868.5,2026-07-03


Las dos filas son idénticas **salvo `cargado_en`**: una se cargó el 1 de
julio y la otra el 3. Es el rastro clásico de un lote de ETL que se ejecutó
dos veces.

Y esto tiene una consecuencia incómoda:

In [ ]:
print(f"drop_duplicates() sin argumentos deja: {len(hechos.drop_duplicates()):,} filas")
print("...o sea, no elimina nada.")

drop_duplicates() sin argumentos deja: 12,800 filas
...o sea, no elimina nada.


¿Por qué? Porque `drop_duplicates()` sin argumentos compara **todas** las
columnas, y `cargado_en` es distinta. Para pandas no son duplicados.

Hay que decirle explícitamente **qué columnas definen un asiento único**.

---
## ✏️ 1.7 · TU TURNO · Eliminar los duplicados

Completa el `subset` para quedarte con 12,000 asientos.

> 💡 **Pista:** `drop_duplicates()` sin argumentos compara todas las
> columnas, y dos asientos legítimos podrían coincidir en fecha y monto.
> ¿Qué columna identifica de forma única un asiento contable?

In [ ]:
hechos = hechos.drop_duplicates(subset=["_______"])

print(f"{len(hechos):,} filas")

In [ ]:
verificar("1.7", hechos)

---
## ✏️ 1.8 · TU TURNO · Unir las tres tablas

Para calcular la desviación necesitamos las tres tablas juntas:

```
   hechos_gasto          presupuesto           dim_centro_costo
   (gasto real)     ⋈    (lo autorizado)   ⋈   (responsables)
```

Vamos por partes. Aquí unimos hechos con la dimensión de centros de costo.

> 💡 **Pista:** `.merge()` necesita saber por qué columna unir. ¿Cuál
> comparten `hechos` y `dim_cc`?

> ⚠️ **Aviso:** este ejercicio va a marcar ❌ **aunque lo escribas bien.**
> No te quedes intentando arreglar el código — lee lo que dice el verificador
> y sigue leyendo abajo. El 1.9 lo resuelve.

In [ ]:
hechos_con_dim = hechos.merge(dim_cc, on="____________", how="inner")

print(f"Antes del join:  {len(hechos):,} filas")
print(f"Después:         {len(hechos_con_dim):,} filas")

In [ ]:
verificar("1.8", hechos_con_dim)

### 🔍 Se perdieron 2,400 filas

El verificador te lo dijo: **tu código está bien**. No hay error de sintaxis
que buscar. El problema son los datos.

Un `inner join` solo conserva las filas cuya llave existe en las dos tablas.
Si 2,400 filas desaparecieron, es porque su `centro_costo` **no coincide con
ningún valor** del catálogo. Veamos qué valores hay:

In [ ]:
en_hechos = set(hechos["centro_costo"].unique())
en_catalogo = set(dim_cc["centro_costo"].unique())

print(f"Valores distintos en hechos:    {len(en_hechos)}")
print(f"Valores distintos en catálogo:  {len(en_catalogo)}")
print()
print("Valores de hechos que NO están en el catálogo:")
for v in sorted(en_hechos - en_catalogo)[:10]:
    print(f"   {v!r}")

Valores distintos en hechos:    48
Valores distintos en catálogo:  8

Valores de hechos que NO están en el catálogo:
   '  CALIDAD '
   '  ENERGÍA '
   '  FLETES '
   '  MANTENIMIENTO '
   '  NÓMINA INDIRECTA '
   '  REFACCIONES '
   '  SEGURIDAD '
   '  SISTEMAS '
   ' Calidad'
   ' Energía'


Ahí está. `'  MANTENIMIENTO '`, `'fletes'`, `'Energía '`…

Para un humano son el mismo centro de costo. Para Python son cadenas
distintas: sobran espacios y las mayúsculas no coinciden.

**Este es el error más caro de los cinco**, porque no avisa. El join corrió
sin excepciones y devolvió un resultado que parece bueno. Si calculas la
desviación sobre 9,600 filas en lugar de 12,000, tu reporte está mal y nadie
se va a dar cuenta hasta el comité.

---
## ✏️ 1.9 · TU TURNO · Normalizar el texto y rehacer el join

Hay que quitar los espacios sobrantes y unificar el uso de mayúsculas.

> 💡 **Pista:** `.str.strip()` quita espacios de los extremos y
> `.str.title()` pone cada palabra en Mayúscula Inicial. Aplícalo **en las
> tres tablas**: si normalizas solo un lado, las llaves siguen sin coincidir.

In [ ]:
for tabla in (hechos, presupuesto, dim_cc):
    tabla["centro_costo"] = tabla["centro_costo"].str.______().str.______()

hechos_con_dim = hechos.merge(dim_cc, on="centro_costo", how="inner")

print(f"Después de normalizar: {len(hechos_con_dim):,} filas")

In [ ]:
verificar("1.9", hechos_con_dim)

12,000 filas. Nada se perdió.

> ℹ️ Fíjate en que normalizamos **las tres tablas con la misma regla**. No
> importa cuál sea la forma "correcta" del nombre; importa que las tres
> tablas usen la misma. Es la única manera de que las llaves coincidan.

---
## 🤔 1.10 · Los importes negativos — ¿error o no?

Hasta aquí llevamos cuatro problemas resueltos. El quinto es distinto:
no se arregla con código, se decide con criterio.

In [ ]:
negativos = hechos[hechos["monto"] < 0]

print(f"Asientos con importe negativo: {len(negativos)}")
print(f"Suman: ${negativos['monto'].sum():,.2f}")
print()
negativos[["folio", "fecha", "planta", "centro_costo", "concepto", "monto"]].head(8)

Asientos con importe negativo: 133
Suman: $-616,007.75



,folio,fecha,planta,centro_costo,concepto,monto
25,CHI-202501-006241,2025-01-02,Chihuahua,Mantenimiento,Mantenimiento preventivo,-4627.50
102,CHI-202501-006361,2025-01-05,Chihuahua,Seguridad,Mantenimiento preventivo,-2467.50
191,IRA-202501-009181,2025-01-08,Irapuato,Fletes,Mantenimiento preventivo,-4257.50
390,CHI-202501-006301,2025-01-17,Chihuahua,Fletes,Mantenimiento preventivo,-3910.75
480,IRA-202501-009121,2025-01-20,Irapuato,Mantenimiento,Mantenimiento preventivo,-10158.25
556,IRA-202501-009241,2025-01-23,Irapuato,Seguridad,Mantenimiento preventivo,-1140.25
828,CHI-202502-006481,2025-02-08,Chihuahua,Refacciones,Mantenimiento preventivo,-7171.00
917,IRA-202502-009301,2025-02-11,Irapuato,Energía,Mantenimiento preventivo,-4760.75


### ✍️ Tu respuesta

Un cargo negativo puede ser dos cosas muy distintas:

* un **error de captura** — alguien puso el signo al revés, o
* un **contra-asiento legítimo** — una nota de crédito, la devolución de una
  refacción, el reverso de una póliza mal aplicada.

**Responde en esta celda** (doble clic para escribir):

1. ¿Qué te hace pensar que estos son una cosa o la otra? Mira los conceptos
   y los montos.
2. ¿Los eliminarías del análisis? Explica qué pasaría con la desviación de
   Hermosillo si los borraras.
3. Si fueras la persona que firma el reporte, ¿qué necesitarías confirmar
   antes de decidir?

> *Escribe aquí tu respuesta…*

---

> 🔑 **La idea que hay que llevarse:** un valor raro no es automáticamente un
> error. "Limpiar datos" no significa borrar lo que se ve raro — significa
> entender por qué está ahí. Borrar estos negativos **inflaría** la
> desviación y te haría reportar un problema más grande del que existe.

---
## 📖 1.11 · Guardar el resultado

Ya tenemos datos en los que podemos confiar. Los guardamos en **Parquet**,
que a diferencia de CSV conserva los tipos de dato: la próxima vez que
abramos este archivo, `monto` seguirá siendo un número y no habrá que
repetir la limpieza.

In [ ]:
DESTINO = RAIZ / "datos" / "checkpoints"
DESTINO.mkdir(parents=True, exist_ok=True)

hechos.to_parquet(DESTINO / "hechos_limpios.parquet", index=False)
presupuesto.to_parquet(DESTINO / "presupuesto_limpio.parquet", index=False)
dim_cc.to_parquet(DESTINO / "dim_centro_costo_limpio.parquet", index=False)

print("Guardado en datos/checkpoints/:")
for f in sorted(DESTINO.glob("*.parquet")):
    print(f"   {f.name:<38} {f.stat().st_size / 1024:>8,.0f} KB")

Guardado en datos/checkpoints/:
   dim_centro_costo_limpio.parquet               3 KB
   hechos_limpios.parquet                      167 KB
   hechos_variance.parquet                      16 KB
   presupuesto_limpio.parquet                    4 KB


---

# ✅ Bloque 1 terminado

| Lo que encontramos | Cómo se detectaba | Cómo se arregló |
|---|---|---|
| ① Archivo en otra codificación | truena al abrir | `encoding="cp1252"` |
| ② Importes como texto | `.info()` dice `str` | quitar `.`, cambiar `,` por `.` |
| ③ 800 asientos duplicados | folios repetidos | `drop_duplicates(subset=["folio"])` |
| ④ Llaves de texto sucias | el join pierde filas | `.str.strip().str.title()` |
| ⑤ Importes negativos | criterio, no código | **no** se borran |

Los cuatro primeros tenían arreglo. El quinto era una trampa.

---

## 🎯 Ahora ve al reto — Paso 1

Abre **`04_reto_final.ipynb`** y haz el **Paso 1** (5 minutos).

Es otro dataset, con los mismos cinco problemas y las mismas técnicas. Pero
lee la pista con cuidado: **no están necesariamente en las mismas tablas.**

Cuando termines, seguimos con `02_sql_con_duckdb.ipynb`.